In [0]:
%run ./connectionNotebook

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, row_number
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@adlsg2rag.dfs.core.windows.net/sqlserver/sales/load_date=2026-03-13/"

In [0]:
df = spark.read.format("parquet").load(bronze_path)
df.printSchema()

In [0]:
# PK validation
df_clean = df.filter(col("SaleID").isNotNull() & (col("SaleID") != 0))

In [0]:
# Date validation
df_clean = df_clean.withColumn("SaleDate", to_date(col("SaleDate")))
df_clean = df_clean.filter(col("SaleDate").isNotNull())

In [0]:
# Processing timestamp
df_clean = df_clean.withColumn("processed_ts", current_timestamp())

In [0]:
# Duplicate check
w = Window.partitionBy("SaleID").orderBy(col("processed_ts").desc())
df_clean = (
    df_clean.withColumn("rn", row_number().over(w))
            .filter(col("rn") == 1)
            .drop("rn")
)

In [0]:
# Select and rename columns
df_clean = df_clean.select(
    col("SaleID").alias("src_SaleID"),
    col("Quantity").alias("src_Quantity"),
    col("SaleDate").alias("src_SaleDate"),
    col("TotalAmount").alias("src_TotalAmount"),
    col("processed_ts")
)


In [0]:
catalog_name = 'adbrag'
schema_name = 'silver'

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.sales_silver
(
    src_SaleID INT,
    src_Quantity INT,
    src_SaleDate DATE,
    src_TotalAmount DECIMAL(10,2),
    processed_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (src_SaleID)
""")


In [0]:
df_clean.write.mode("overwrite").format("delta").saveAsTable(
    f"{catalog_name}.{schema_name}.sales_silver"
)